# 03 · Data structures

Choose a container, inspect it, and transform its contents. This chapter develops the material in *3_data_structures.pdf* (pages 1–54) for current Python.

**Before you start:** know variables, conditions, and `for` loops. Run cells from top to bottom in a fresh Python 3 kernel. Examples contain their own data, and expected errors are caught. Exercise cells are safe placeholders; replace their comments with your work. Solutions are in `../solutions/03_data_structures_solutions.ipynb`.


## What you will be able to do

- Use lists, tuples, dictionaries, and sets for different kinds of data.
- Explain mutation, aliasing, shallow copies, and hashability.
- Combine unpacking, `enumerate`, `zip`, `reversed`, and `sorted`.
- Translate short loops into list, dictionary, and set comprehensions.

| Container | Main purpose | Ordered? | Mutable? |
|---|---|---|---|
| `list` | A sequence that can change | Yes, positional | Yes |
| `tuple` | A fixed sequence of references | Yes, positional | No |
| `dict` | Map unique keys to values | Yes, insertion order | Yes |
| `set` | Distinct elements and membership | No positional order | Yes |

“Mutable” describes the container itself. Objects stored inside it may have their own mutability rules.


## 1. Lists: ordered, mutable sequences

Square brackets create a list. Elements may have different types, although a consistent element type often makes later processing simpler. An empty list is still a list. `append` adds **one object**; `extend` consumes an iterable and adds its individual elements.


In [ ]:
empty = []
letters = ["a", "b", "c", "d"]
numbers = [2, 3, 5]
mixed = [4, 5, "seconds"]
numbers.append(7)
numbers.extend([11, 13])
print(empty, letters, mixed)
print(numbers)  # [2, 3, 5, 7, 11, 13]

with_nested_item = [1, 2]
with_nested_item.append([3, 4])
print(with_nested_item)  # [1, 2, [3, 4]]


### Indexing and slicing

Indices start at zero; negative indices count from the end. A slice `start:stop:step` excludes `stop`. Slicing produces a new list, and out-of-range slice bounds are clipped. A single invalid index raises `IndexError`.


In [ ]:
values = [10, 20, 30, 40, 50]
print(values[0], values[-1])       # 10 50
print(values[1:-1])               # [20, 30, 40]
print(values[:3], values[::2])    # [10, 20, 30] [10, 30, 50]
print(values[::-1], values[99:])  # [50, 40, 30, 20, 10] []
try:
    print(values[99])
except IndexError:
    print("Expected IndexError: there is no element at index 99.")


### Change a list in place

Item assignment replaces one reference. Slice assignment may replace several elements and change the list's length. `insert(index, value)` inserts before an index, `remove(value)` removes the first equal value, and `del` deletes an item or slice.


In [ ]:
queue = ["Ada", "Ben", "Cara"]
queue[1] = "Bo"
queue.insert(1, "Dee")
print(queue)  # ['Ada', 'Dee', 'Bo', 'Cara']
queue[2:] = ["Eli", "Fay", "Gia"]
queue.remove("Dee")
del queue[-1]
print(queue)  # ['Ada', 'Eli', 'Fay']

next_person = queue.pop(0)
last_person = queue.pop()
print(next_person, last_person, queue)  # Ada Fay ['Eli']
queue.clear()
print(queue)  # []


### Query lists and handle missing values

`len` reports length; `in` tests membership; `count` counts equal values. `index(value, start, stop)` finds the first matching position in the search interval. It and `remove` raise `ValueError` when absent; `pop` raises `IndexError` when its index is invalid.

These operations apply to many **containers and sequences**. Do not assume every iterable supports `len` or indexing: for example, a generator does not.


In [ ]:
colors = ["red", "blue", "red", "green"]
print(len(colors), "blue" in colors, "yellow" not in colors)  # 4 True True
print(colors.count("red"), colors.index("red", 1))  # 2 2
print("y" in "python")  # True
for operation in (lambda: colors.index("pink"), lambda: colors.remove("pink")):
    try:
        operation()
    except ValueError:
        print("Expected ValueError: pink is not in this list.")
try:
    [].pop()
except IndexError:
    print("Expected IndexError: cannot pop from an empty list.")


### Sorting and reversing: change the source or create a result?

`list.sort()` and `list.reverse()` mutate the list and return `None`. `sorted(iterable)` builds a new list. Sorting is **stable**: elements with equal keys retain their previous order. Supply a `key` to select what to compare and `reverse=True` for descending order.


In [ ]:
names = ["Zoe", "Al", "Bo", "Ada"]
by_length = sorted(names, key=len)
print(names)      # ['Zoe', 'Al', 'Bo', 'Ada'] — unchanged
print(by_length)  # ['Al', 'Bo', 'Zoe', 'Ada'] — equal lengths stay in order
result = names.sort(key=len, reverse=True)
print(names, result)  # ['Zoe', 'Ada', 'Al', 'Bo'] None
names.reverse()
print(names)  # ['Bo', 'Al', 'Ada', 'Zoe']


### Nested lists, aliases, and shallow copies

A list can hold other lists. `alias = original` gives the same object another name. `original.copy()` (or `original[:]`) makes a **new outer list**, but its elements still refer to the same objects. Copying a nested list therefore does not automatically make its rows independent.

For a matrix of scalar values, `[row.copy() for row in matrix]` copies the outer list and each row. For arbitrary nested structures, the standard library provides `copy.deepcopy`; the right copying strategy depends on which sharing you intend.


In [ ]:
matrix = [[1, 2], [3, 4]]
alias = matrix
shallow = matrix.copy()
independent_rows = [row.copy() for row in matrix]
print(matrix[1][0], matrix[0][1:])  # 3 [2]
print(alias is matrix, shallow is matrix, shallow[0] is matrix[0])  # True False True
shallow[0].append(9)
print(matrix)            # [[1, 2, 9], [3, 4]]
print(independent_rows)  # [[1, 2], [3, 4]]

shared_rows = [[0] * 2] * 2
shared_rows[0][0] = 7
print(shared_rows)  # [[7, 0], [7, 0]] — both entries refer to one row!
separate_rows = [[0] * 2 for _ in range(2)]
separate_rows[0][0] = 7
print(separate_rows)  # [[7, 0], [0, 0]]


### Exercise 03.1 · Edit a queue and copy a matrix

1. Start with `queue = ["Ada", "Ben", "Cara"]`. Append `"Dee"`, insert `"Eli"` before `"Ben"`, remove `"Cara"`, and save the person removed by `pop(0)` as `served`.
2. Build `reversed_queue` without changing `queue`.
3. Copy every row of `matrix = [[1, 2], [3, 4]]` into `matrix_copy`. Change the first copied value to `99`, leaving `matrix` unchanged.

Expected: `served == "Ada"`, `queue == ["Eli", "Ben", "Dee"]`, `reversed_queue == ["Dee", "Ben", "Eli"]`, and `matrix_copy == [[99, 2], [3, 4]]`. Explain why `matrix.copy()` alone would not meet step 3.


In [ ]:
# Exercise 03.1
# TODO: Build and edit queue, then create reversed_queue.
# TODO: Copy the matrix rows and change only matrix_copy.
# TODO: Add assertions for the expected results and original matrix.


## 2. Tuples: fixed sequences of references

A tuple is immutable: its element references cannot be replaced, removed, or appended. Indexing, slicing, `len`, and membership still work. The **comma** constructs a tuple; parentheses usually group it. A singleton needs a trailing comma: `("value",)`.

Use tuples for a fixed group of values, such as `(latitude, longitude)`, or multiple return values. A tuple is hashable **only if all of its elements are hashable**. A tuple containing a list is not a valid dictionary key.


In [ ]:
record = (7, "Ada", "Vienna")
print(record[1], record[:2], "Ada" in record, len(record))
print(type(()).__name__, type(("value",)).__name__, type(("value")).__name__)
try:
    record[0] = 8
except TypeError:
    print("Expected TypeError: tuple elements cannot be assigned.")

nested_tuple = ([1, 2], "measurements")
nested_tuple[0].append(3)
print(nested_tuple)  # ([1, 2, 3], 'measurements') — the inner list can change
try:
    hash(nested_tuple)
except TypeError:
    print("Expected TypeError: a tuple containing a list is unhashable.")


### Packing, unpacking, and swapping

`point = 3, 4` packs two values. `x, y = point` unpacks them. The number of targets must match, unless one starred target collects the remaining values into a list. Unpacking works with other iterables too.

In `x, y = y, x`, Python evaluates the right side before assigning the left side. This expresses a swap directly, without a temporary variable or integer-specific XOR operations.


In [ ]:
point = 3, 4
x, y = point
x, y = y, x
print(point, x, y)  # (3, 4) 4 3
first, *middle, last = [10, 20, 30, 40]
print(first, middle, last)  # 10 [20, 30] 40
try:
    a, b = (1, 2, 3)
except ValueError:
    print("Expected ValueError: too many values for two targets.")


### Fibonacci updates and indexed iteration

Simultaneous assignment is useful when a new value depends on two old values. Fibonacci numbers start with `0, 1`; each later number is the sum of the two before it. `a, b = b, a + b` updates both names using the old values.

Use `enumerate(sequence, start=...)` when you need an element and its position. Use `range(len(sequence))` when the actual algorithm needs indices, such as assigning to positions or comparing neighbors.


In [ ]:
a, b = 0, 1
fibonacci = []
for _ in range(8):
    fibonacci.append(a)
    a, b = b, a + b
print(fibonacci)  # [0, 1, 1, 2, 3, 5, 8, 13]

for position, color in enumerate(["red", "green", "blue"], start=1):
    print(position, color)
# 1 red
# 2 green
# 3 blue


### Exercise 03.2 · Fibonacci records

Set `n = 7`. Use simultaneous assignment to build the first `n` Fibonacci numbers, starting with `0`. Then use `enumerate` and unpacking to build `records`, a list of `(index, value)` tuples with indices starting at zero.

Expected: `records == [(0, 0), (1, 1), (2, 1), (3, 2), (4, 3), (5, 5), (6, 8)]`. Repeat with `n = 0` and `n = 1`; expect `[]` and `[(0, 0)]`. A loop is enough; defining a helper function is optional.


In [ ]:
# Exercise 03.2
# TODO: Build Fibonacci numbers and enumerate them into records.
# TODO: Check n = 7, n = 0, and n = 1.


## 3. Dictionaries: map unique keys to values

A dictionary maps **hashable keys** to arbitrary values. Common keys include strings, numbers, and tuples of hashable elements. Lists, dictionaries, and mutable sets cannot be keys.

Current Python preserves **insertion order**. Assigning an existing key changes its value without moving it. Deleting and reinserting a key adds it at the end. Insertion order is not sorted order, and dictionary equality compares key/value contents rather than insertion order.


In [ ]:
empty_mapping = {}
counts = dict(one=1, two=2, three=3)
print(empty_mapping == dict())  # True
print(counts == {"three": 3, "one": 1, "two": 2})  # True
counts["two"] = 22
counts["four"] = 4
print(counts["one"], list(counts))  # 1 ['one', 'two', 'three', 'four']
del counts["one"]
counts["one"] = 1
print(list(counts))  # ['two', 'three', 'four', 'one']
locations = {(48.2, 16.4): "Vienna"}
print(locations[(48.2, 16.4)])  # Vienna
try:
    locations[[48.2, 16.4]] = "invalid"
except TypeError:
    print("Expected TypeError: lists cannot be dictionary keys.")


### Missing keys and defaults

`mapping[key]` raises `KeyError` for an absent key. `mapping.get(key)` returns `None`; supply a second argument for a different default. `get` does **not** insert the default into the dictionary.

`key in mapping` checks keys, not values. Distinguish an absent key from a present key whose value is `None` by checking membership.


In [ ]:
courses = {"CS": [106, 107, 110], "MATH": [51, 113], "PHIL": None}
print(courses.get("CS"))              # [106, 107, 110]
print(courses.get("ENGLISH"))         # None
print(courses.get("ENGLISH", []))     # []
print("ENGLISH" in courses, "PHIL" in courses)  # False True
print([51, 113] in courses.values())   # True
try:
    print(courses["COMPSCI"])
except KeyError:
    print("Expected KeyError: COMPSCI is absent.")


### Dynamic views, iteration, and deletion

`keys()`, `values()`, and `items()` return live views. A saved view reflects later changes; `list(view)` creates a snapshot. Iterate over `items()` and unpack its `(key, value)` pairs.

`pop(key, default)` removes and returns a value; without a default, a missing key raises `KeyError`. `popitem()` removes the **last inserted** pair in current Python and raises `KeyError` on an empty dictionary. Do not add or delete keys while iterating a live dictionary view; iterate over a snapshot when structural changes are needed.


In [ ]:
scores = {"Ada": 90, "Ben": 75}
keys_view = scores.keys()
keys_snapshot = list(keys_view)
scores["Cara"] = 82
print(list(keys_view), keys_snapshot)  # ['Ada', 'Ben', 'Cara'] ['Ada', 'Ben']
print(("Ada", 90) in scores.items())   # True
for name, score in scores.items():
    print(name, score)
print(scores.pop("missing", 0))  # 0
print(scores.pop("Ben"))         # 75
print(scores.popitem())          # ('Cara', 82)
del scores["Ada"]
try:
    scores.popitem()
except KeyError:
    print("Expected KeyError: the dictionary is empty.")


### Copies and grouping

`dict.copy()` is shallow, like `list.copy()`. `clear()` empties a dictionary. `setdefault(key, default)` returns the existing value, or inserts and returns the default when the key is absent; it is convenient for grouping. `update` adds new pairs and replaces existing values.


In [ ]:
original = {"CS": [106]}
shallow = original.copy()
shallow["CS"].append(107)
print(original)  # {'CS': [106, 107]}
shallow.clear()
print(original, shallow)  # {'CS': [106, 107]} {}

by_city = {}
for name, city in [("Ada", "Vienna"), ("Bo", "Graz"), ("Cara", "Vienna")]:
    by_city.setdefault(city, []).append(name)
print(by_city)  # {'Vienna': ['Ada', 'Cara'], 'Graz': ['Bo']}
settings = {"theme": "light", "size": 12}
settings.update({"theme": "dark", "language": "en"})
print(settings)  # {'theme': 'dark', 'size': 12, 'language': 'en'}


### Exercise 03.3 · Count and group names

For `visits = [("Ada", "Vienna"), ("Bo", "Graz"), ("Ada", "Vienna"), ("Cara", "Vienna")]`:

1. Build `visit_counts`, counting each person's visits with `get` and a default of zero.
2. Build `people_by_city`, storing each city's visitors in encounter order, including repeat visits.
3. Retrieve the number of `"Dee"` visits as zero without inserting a new key.

Expected: `visit_counts == {"Ada": 2, "Bo": 1, "Cara": 1}` and `people_by_city == {"Vienna": ["Ada", "Ada", "Cara"], "Graz": ["Bo"]}`. Empty input should produce two empty dictionaries.


In [ ]:
# Exercise 03.3
# TODO: Count visits and group visitor names by city.
# TODO: Check a missing name and empty input.


## 4. Sets: distinct, hashable elements

A set removes duplicates and supports membership tests. Its elements must be hashable. Use `set()` for an empty set: `{}` creates a dictionary. Sets have no meaningful positional order, so do not index them or rely on iteration order. Print `sorted(a_set)` when a reproducible display matters.

Set and dictionary membership are **average-case O(1)** under usual hash-table assumptions; list membership is O(n). Hashing cost, collisions, and input size still matter. Turning a list into a set also takes work, so it is particularly useful for repeated lookups, uniqueness, and set algebra.


In [ ]:
basket = set(["apple", "orange", "apple", "pear"])
print(sorted(basket), len(basket), "orange" in basket)
# ['apple', 'orange', 'pear'] 3 True
basket.add("banana")
basket.remove("orange")
basket.discard("missing")  # No error if absent.
print(sorted(basket))  # ['apple', 'banana', 'pear']
try:
    basket.remove("missing")
except KeyError:
    print("Expected KeyError: remove needs a present element.")
removed = basket.pop()  # Which element is removed is not promised.
print(removed not in basket, len(basket))  # True 2
basket.clear()
print(basket == set())  # True
try:
    {[]}
except TypeError:
    print("Expected TypeError: set elements must be hashable.")


### Set algebra and subset tests

For two sets, `|` is union, `&` intersection, `-` difference, and `^` symmetric difference (in exactly one input). `<=` tests subset, `<` proper subset, and `isdisjoint` tests whether there is no overlap.

A word contains only allowed letters when `set(word) <= allowed`. This test ignores order and repetitions. An empty set is a subset of every set, so an empty word passes this rule; add an explicit nonempty check if your application needs one.


In [ ]:
a = set("abracadabra")
b = set("alacazam")
print(sorted(a | b))  # ['a', 'b', 'c', 'd', 'l', 'm', 'r', 'z']
print(sorted(a & b))  # ['a', 'c']
print(sorted(a - b))  # ['b', 'd', 'r']
print(sorted(a ^ b))  # ['b', 'd', 'l', 'm', 'r', 'z']
print({"a", "c"} <= a, {"a", "c"} < a, a.isdisjoint(b))  # True True False
allowed = set("BCDGIJLMNOPSUVWZ")
for word in ["BOW", "WIND", "TREE", ""]:
    print(repr(word), set(word) <= allowed)
# 'BOW' True; 'WIND' True; 'TREE' False; '' True


### Exercise 03.4 · Compare course enrollments

Use `python_students = ["Ada", "Bo", "Cara", "Ada"]` and `statistics_students = ["Bo", "Dee"]`. Create sets and calculate students in both courses, either course, Python only, and exactly one course. Present each result as a sorted list.

Expected: both `['Bo']`; either `['Ada', 'Bo', 'Cara', 'Dee']`; Python only `['Ada', 'Cara']`; exactly one `['Ada', 'Cara', 'Dee']`. Also test two empty enrollments and two identical enrollments. Why do duplicate signups disappear?


In [ ]:
# Exercise 03.4
# TODO: Convert enrollments to sets and use &, |, -, and ^.
# TODO: Sort display results and check empty/identical inputs.


## 5. Loop over what you need

- `enumerate` supplies index/element pairs.
- `dict.items()` supplies key/value pairs.
- `zip` supplies corresponding items from several iterables. By default it stops at the shortest input; `strict=True` raises `ValueError` for unequal lengths.
- `reversed(sequence)` iterates backward without changing the sequence.
- `sorted(iterable, key=..., reverse=...)` creates a sorted list.

`zip`, `enumerate`, and `reversed` return iterators, so `list(...)` is useful when you need to inspect all their results. Once consumed, an iterator is exhausted.


In [ ]:
questions = ["name", "quest", "favorite color"]
answers = ["Lancelot", "Find the grail", "Blue"]
for question, answer in zip(questions, answers, strict=True):
    print(f"{question}: {answer}")
print(list(zip([1, 2, 3], ["a", "b"])))  # [(1, 'a'), (2, 'b')]
try:
    list(zip([1, 2, 3], ["a", "b"], strict=True))
except ValueError:
    print("Expected ValueError: paired inputs have different lengths.")
print(list(reversed(range(1, 10, 2))))  # [9, 7, 5, 3, 1]
print(sorted(["pear", "banana", "pear", "apple"]))
# ['apple', 'banana', 'pear', 'pear'] — sorting does not deduplicate
pairs = zip([1, 2], ["a", "b"])
print(list(pairs), list(pairs))  # [(1, 'a'), (2, 'b')] []


## 6. Comprehensions: describe a transformation

`[expression for item in iterable]` creates a list. Add `if condition` after the loop to filter the input. Read it as “produce this expression for each item that passes the condition.”

A comprehension is useful when the transformation fits clearly in one expression. Use a regular loop for several steps, complex decisions, or actions whose main purpose is a side effect.


In [ ]:
squares_loop = []
for number in range(100):
    squares_loop.append(number ** 2)
squares = [number ** 2 for number in range(100)]
assert squares == squares_loop
print(squares[:5] + squares[-5:])  # [0, 1, 4, 9, 16, 9025, 9216, 9409, 9604, 9801]
print([2 * x + 1 for x in [0, 1, 2, 3]])  # [1, 3, 5, 7]
print([x % 3 == 0 for x in [3, 5, 9, 8]])  # [True, False, True, False]
fruits = ["apple", "orange", "pear"]
print([fruit[0].upper() for fruit in fruits])  # ['A', 'O', 'P']
print([fruit for fruit in fruits if len(fruit) < 6])  # ['apple', 'pear']
print([(fruit, len(fruit)) for fruit in fruits])
# [('apple', 5), ('orange', 6), ('pear', 4)]


### Nested, dictionary, and set comprehensions

Multiple `for` clauses follow the same left-to-right order as nested loops. `{key: value for ...}` makes a dictionary; `{expression for ...}` makes a set. A dictionary keeps only the last value produced for a repeated key; a set keeps distinct results.

Inverting a dictionary with `{value: key ...}` is safe only when its values are hashable and you accept losing information when values repeat. Avoid clever comprehensions that hide these decisions.


In [ ]:
pairs = [(i, j) for i in range(4) for j in range(i)]
print(pairs)  # [(1, 0), (2, 0), (2, 1), (3, 0), (3, 1), (3, 2)]
rows = [[1, 2], [3, 4]]
print([value for row in rows for value in row])  # [1, 2, 3, 4]
print({x: x ** 2 for x in range(4)})  # {0: 0, 1: 1, 2: 4, 3: 9}
words = ["Level", "python", "level", "Radar"]
palindromes = {word.lower() for word in words if word.lower() == word.lower()[::-1]}
print(sorted(palindromes))  # ['level', 'radar']
capitals = {"Austria": "Vienna", "France": "Paris"}
print({city: country for country, city in capitals.items()})
repeated = {"Ada": "A", "Bo": "A"}
print({grade: name for name, grade in repeated.items()})  # {'A': 'Bo'}


### Exercise 03.5 · A comprehension toolkit

Using `words = ["Apple", "pear", "BANANA", "pear", "", "kiwi"]`, create:

1. `normalized`: lowercase words, preserving order and duplicates.
2. `long_words`: lowercase words with at least five characters.
3. `lengths`: a dictionary from each nonempty lowercase word to its length.
4. `initials`: the set of uppercase first letters of nonempty words.

Expected: `normalized == ['apple', 'pear', 'banana', 'pear', '', 'kiwi']`, `long_words == ['apple', 'banana']`, `lengths == {'apple': 5, 'pear': 4, 'banana': 6, 'kiwi': 4}`, and `initials == {'A', 'P', 'B', 'K'}`. Use comprehensions and make empty input work too.


In [ ]:
# Exercise 03.5
# TODO: Write two list comprehensions, one dict comprehension, and one set comprehension.
# TODO: Avoid indexing an empty string, and test an empty input list.


### Exercise 03.6 · Build a small result table

Use `names = ["Ada", "Bo", "Cara"]` and `scores = [88, 65, 88]`.

1. Pair corresponding values with `zip(..., strict=True)` and build a list of `(name, score)` tuples.
2. Sort records by score descending. Preserve input order when scores tie.
3. Build a `passed` dictionary for scores of at least `70`.
4. Number the sorted records from `1` using `enumerate`.

Expected sorted records: `[('Ada', 88), ('Cara', 88), ('Bo', 65)]`; `passed == {'Ada': 88, 'Cara': 88}`; numbered records start with `(1, ('Ada', 88))`. Verify empty lists work and unequal lengths raise a caught `ValueError`.


In [ ]:
# Exercise 03.6
# TODO: Pair, sort, filter, and number the records.
# TODO: Check stable tie order, empty input, and a caught length mismatch.


## Check your understanding

Before opening the solutions, explain these in your own words:

- Why does `append([3, 4])` differ from `extend([3, 4])`?
- Why can a tuple contain a mutable list but fail as a dictionary key?
- How do a live dictionary view and a list snapshot differ?
- What information is lost when you convert a list to a set or invert a dictionary with repeated values?
- When does `sorted` preserve an existing order, and when does `zip` silently leave values unpaired?

The six exercises have complete, independently runnable answers in [the solutions notebook](../solutions/03_data_structures_solutions.ipynb). Standalone examples are in [examples/03_data_structures.py](../examples/03_data_structures.py). Continue with [04 · Functions](04_functions.ipynb).
